## FIFA WORLD CUP FINAL ANALISIS AND WINNER PREDICTION
### This notebook scrapes team stadistics from FIFA's Website for Spain and Argentica national fotbal teams to predict the 2026 FIFA world cup champion

In [37]:
#Install required libraries 
#!pip install selenium 
# Import required libraries 
import pandas as pd
import numpy as np
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time 
from functools import reduce
from scipy.stats import poisson

In [2]:
# Data ingestion from FIFA's website 
# Initiate driver
driver = webdriver.Chrome()
#Define URL for FIFA's website
URL = "https://www.fifa.com/es/tournaments/mens/worldcup/canadamexicousa2026/statistics/team-statistics"
# Navigate driver to website 
driver.get(URL)
#Wait for website to load 
WebDriverWait(
    driver, 10).until(EC.presence_of_element_located((By.CSS_SELECTOR, ".cell-label"))
)
#Pulls headers data from website
header_elements = driver.find_elements(By.CSS_SELECTOR, "thead .cell-label")
headers = [el.text for el in header_elements]
#Pulls table body from website 
body = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
teams = ["Argentina", "España"]
for x in body:
    row_data = x.text.split()
    if len(row_data)> 1:
        if row_data[1] in teams: 
            table = dict(zip(headers, row_data))
            print ("Equipo encontrado", table)
#Close driver 
driver.quit()


Equipo encontrado {'Puesto': '1', 'Equipo': 'Argentina', 'Goles': '19', 'Asistencias': '12', 'Remate': '113', 'Remate entre los tres palos': '46', 'Tiros fuera': '48', 'Efectividad en los remates Tipo (%)': '17', 'Tiros dentro del área': '65', 'Tiros fuera del área': '48', 'Remates de cabeza': '22', 'Goles prev.': '15.38', 'Efect. en goles prev.': '1.24x', 'Saque de esquina': '37', 'Posesión del balón (%)': '55'}
Equipo encontrado {'Puesto': '5', 'Equipo': 'España', 'Goles': '13', 'Asistencias': '9', 'Remate': '120', 'Remate entre los tres palos': '42', 'Tiros fuera': '53', 'Efectividad en los remates Tipo (%)': '11', 'Tiros dentro del área': '71', 'Tiros fuera del área': '49', 'Remates de cabeza': '17', 'Goles prev.': '14.96', 'Efect. en goles prev.': '0.87x', 'Saque de esquina': '45', 'Posesión del balón (%)': '58'}


In [3]:
#Tabs list accoirding to FIFA´s website 
tabs = ["Ataque", "Distribución", "Defensa", "Disciplina","Portería", "Movimiento", "Físico"]
#Funtion that pulls data from each tab and concatenates in master df 
def fetch_tab_data(driver, tab_name, first_tab= False):
    #create empty list to store results
    tab_data = [] 
    if not first_tab:
        boton = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.XPATH, f"//*[contains(text(), '{tab_name}')]"))
        )
        boton.click()
        time.sleep(5)
        pass
    #Wait for website to load 
    WebDriverWait(
        driver, 15).until(EC.presence_of_element_located((By.CSS_SELECTOR, "tbody tr"))
    )
    #Pulls headers data from website
    header_elements = driver.find_elements(By.CSS_SELECTOR, "thead .cell-label")
    headers = [el.text for el in header_elements]
    #Pulls table body from website 
    body = driver.find_elements(By.CSS_SELECTOR, "tbody tr")
    teams = ["Argentina", "España"]
    for x in body:
        row_data = x.text.split()
        if len(row_data)> 1:
            if row_data[1] in teams: 
                table = dict(zip(headers, row_data))
                print ("Equipo encontrado", f"{table["Equipo"]} agregada a tab_data")
                tab_data.append(table)
    return tab_data




In [4]:
#THIS CELL PULLS DATA FROM ALL TABS AND CONCENTRATES IN A MASTER DICT TO CONVER INTO PANDAS DF  
#Tabs list accoirding to FIFA´s website 
tabs = ["Ataque", "Distribución", "Defensa", "Disciplina","Portería", "Movimiento", "Físico"]
master_data = {}
#Init driver 
driver= webdriver.Chrome() 
driver.maximize_window()
driver.get(URL)
#Remove cookies banner
try:
        cookies_button=WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Rechazarlas todas')]"))
        )
        cookies_button.click()
        print("💥 Banner overcome")
        time.sleep(1)
except Exception as e:
        print("ℹ️ No banner, continue scrapping")
        
#Loop to fetch data from all tabs
for i, tab in enumerate(tabs):
    ftab = (i== 0) 
    
    #Scrap data 
    scraped_data = fetch_tab_data (driver, tab_name=tab, first_tab=ftab)
    master_data[tab] = scraped_data
    
print("All FIFA data succesufly retrieved.")
    

driver.quit()

💥 Banner overcome
Equipo encontrado Argentina agregada a tab_data
Equipo encontrado España agregada a tab_data
Equipo encontrado Argentina agregada a tab_data
Equipo encontrado España agregada a tab_data
Equipo encontrado Argentina agregada a tab_data
Equipo encontrado España agregada a tab_data
Equipo encontrado Argentina agregada a tab_data
Equipo encontrado España agregada a tab_data
Equipo encontrado España agregada a tab_data
Equipo encontrado Argentina agregada a tab_data
Equipo encontrado España agregada a tab_data
Equipo encontrado Argentina agregada a tab_data
Equipo encontrado España agregada a tab_data
Equipo encontrado Argentina agregada a tab_data
All FIFA data succesufly retrieved.


In [5]:
print(master_data["Distribución"])

[{'Puesto': '1', 'Equipo': 'Argentina', 'Pase': '4772', 'Pases completados': '4324', 'Precisión en los pases (%)': '91', 'Centro': '102', 'Precisión en los centros (%)': '33', 'Regates completados': '50', 'Intentos de ruptura de líneas': '122', 'Acierto en las rupturas de líneas ((%))': '60', 'Cambios de orientación intentados': '34', 'Acierto en los cambios de orientación ((%))': '94'}, {'Puesto': '2', 'Equipo': 'España', 'Pase': '4592', 'Pases completados': '4156', 'Precisión en los pases (%)': '91', 'Centro': '154', 'Precisión en los centros (%)': '21', 'Regates completados': '61', 'Intentos de ruptura de líneas': '175', 'Acierto en las rupturas de líneas ((%))': '65', 'Cambios de orientación intentados': '40', 'Acierto en los cambios de orientación ((%))': '95'}]


In [8]:
#Transforms dict to pandas df 
df_ataque = pd.DataFrame(master_data["Ataque"])
df_defensa = pd.DataFrame(master_data["Defensa"])
df_distribucion = pd.DataFrame(master_data["Distribución"])
df_disciplina = pd.DataFrame(master_data["Disciplina"])
df_porteria = pd.DataFrame(master_data["Portería"])
df_movimiento = pd.DataFrame(master_data["Movimiento"])
df_fisico = pd.DataFrame(master_data["Físico"])



In [14]:
df_final = df_ataque.merge(
    df_distribucion.drop(columns=["Puesto"]), on="Equipo"
).merge(
    df_defensa.drop(columns=["Puesto"]), on="Equipo"
).merge(
    df_disciplina.drop(columns=["Puesto"]), on="Equipo"
).merge(
    df_porteria.drop(columns=["Puesto"]), on="Equipo"
).merge(
    df_movimiento.drop(columns=["Puesto"]), on="Equipo"
).merge(
    df_fisico.drop(columns=["Puesto"]), on="Equipo"
)

In [29]:
#Final DF INDEX
df_final.dtypes

Puesto                                                 object
Equipo                                                 object
Goles                                                  object
Asistencias                                            object
Remate                                                 object
Remate entre los tres palos                            object
Tiros fuera                                            object
Efectividad en los remates Tipo (%)                    object
Tiros dentro del área                                  object
Tiros fuera del área                                   object
Remates de cabeza                                      object
Goles prev.                                            object
Efect. en goles prev.                                  object
Saque de esquina                                       object
Posesión del balón (%)                                 object
Pase                                                   object
Pases co

In [35]:
#Create a copy from df_final to modelate Feature selection and normalization for ofensive and defensive score
model_df = df_final.copy() 
#Sets team as index for df 
model_df = model_df.set_index("Equipo")
#drops unnecessary column 
model_df = model_df.drop("Efect. en goles prev.", axis=1)
#Converts dtype to numeric 
model_df = model_df.apply(pd.to_numeric)

In [36]:
#This block contains feature enginering, data normalization and creates an offensive/defensive score for each team
#Creates new columns for normalized features to determine ofensive score 
model_df['Goles_norm'] = model_df['Goles'] / model_df['Goles'].max()
model_df['Asistencias_norm'] = model_df['Asistencias'] / model_df['Asistencias'].max()
model_df['Remate entre los tres palos_norm'] = model_df['Remate entre los tres palos'] / model_df['Remate entre los tres palos'].max()
model_df['Efectividad en los remates Tipo (%)_norm'] = model_df['Efectividad en los remates Tipo (%)'] / model_df['Efectividad en los remates Tipo (%)'].max()
model_df['Goles prev._norm'] = model_df['Goles prev.'] / model_df['Goles prev.'].max()
model_df['Remate_norm'] = model_df['Remate'] / model_df['Remate'].max()
#Creates list for ofensive normalized columns 
of_norm_col = ['Goles_norm', 'Asistencias_norm', 'Remate entre los tres palos_norm', 'Efectividad en los remates Tipo (%)_norm', 'Goles prev._norm', 'Remate_norm']
#Calculates offensive score with normalized offensive columns 
model_df['of_score']= model_df[of_norm_col].mean(axis=1) 

#creates new columns for normalized features to determine defensive score
#In this features lower number means a better result
model_df['Goles recibidos_norm'] = 1 - (model_df['Goles recibidos_x'] /  model_df['Goles recibidos_x'].max())
model_df['Tiempo de recuperación del balón (s)_norm'] = 1 - (model_df['Tiempo de recuperación del balón (s)'] / model_df['Tiempo de recuperación del balón (s)'].max())
#In this features a boger number means a better result 
model_df['Pérdidas de balón provocadas_norm'] = model_df['Pérdidas de balón provocadas'] / model_df['Pérdidas de balón provocadas'].max()
model_df['Porterías a cero_norm'] = model_df['Porterías a cero'] / model_df['Porterías a cero'].max()
model_df['Paradas de la portera_norm'] = model_df['Paradas de la portera'] / model_df['Paradas de la portera'].max()
#creates a list of defensive normalized columns
def_norm_col = ['Goles recibidos_norm','Tiempo de recuperación del balón (s)_norm','Pérdidas de balón provocadas_norm','Porterías a cero_norm','Paradas de la portera_norm']
#Calculates deffensive score with normalized deffensive columns 

model_df['def_score'] = model_df[def_norm_col].mean(axis=1)


In [63]:
model_df['def_score']

Equipo
Argentina    0.368416
España       0.819826
Name: def_score, dtype: float64

In [84]:
#This block calculates probability for total number of goals and corners during the match 
#Extract goals lambdas for each team divided by number of matches played during tournament (7)
esp_goal_lambda = model_df.loc['España', 'Goles prev.'] / 7 
arg_goal_lambda = model_df.loc['Argentina', 'Goles prev.'] / 7
#Extracts corner lambdas for each team  divided by number of matches played during tournament (7)
esp_cor_lambda = model_df.loc['España', 'Saque de esquina'] / 7
arg_cor_lambda = model_df.loc['Argentina', 'Saque de esquina'] / 7 
#Calculates adjusted GOAL lambdas considering defensive team scores 
esp_def_score = model_df.loc['España', 'def_score']
arg_def_score = model_df.loc['Argentina', 'def_score']

#Calibration weight:
def_weight = .30

adj_esp_lambda_goal = esp_goal_lambda * (1 - (arg_def_score) + def_weight)
adj_arg_lambda_goal = arg_goal_lambda * (1 - (esp_def_score) * def_weight) 

#Calculates probability for goals from 1-5 for each team 
prob_goals_esp = [poisson.pmf(k, mu = adj_esp_lambda_goal) for k in range (6)]
prob_goals_arg = [poisson.pmf(k, mu = adj_arg_lambda_goal) for k in range (6)]

#Calculates probability for total match goals 
lambda_goal_total = adj_esp_lambda_goal + adj_arg_lambda_goal

# Calculates Under 2.5 sum of prob of  0, 1 y 2 goals in total
prob_under_2_5 = sum(poisson.pmf(k, mu=lambda_goal_total) for k in [0, 1, 2])

#Over 2.5 is the oporsite, obtain by substracting under vs 1 
prob_over_2_5 = 1 - prob_under_2_5

#Calculates lambda for corners both teams 
total_lambda_cor= esp_cor_lambda + arg_cor_lambda

#Calculates over/ under prob with poisson for 6-10 corners 
#corner lines offered by casino
corner_lines = [6,7,8,9,10]
#creates dict for corner prob results 
corner_probs={}
#loop to calculate probs 
for line in corner_lines:
    cor_prob_under = sum ([poisson.pmf(k, mu=total_lambda_cor) for k in range(line+1)])
    cor_prob_over = 1 - cor_prob_under
    # save results in corner_probs list 
    corner_probs[line] = {
        'Under': cor_prob_under * 100,
        'Over': cor_prob_over * 100
        } 

In [85]:
#This block predicts world cup winner with a Monte Carlo simmulation 
#Define number of simulations 
sim_num = 10000

#Simulates teams goal totals with adjusted lambdas 
goals_sim_arg = np.random.poisson(adj_arg_lambda_goal, sim_num)
goals_sim_esp = np.random.poisson(adj_esp_lambda_goal, sim_num)

#Ponderates goal total simulations to determine winner 
arg_wins = np.sum(goals_sim_arg > goals_sim_esp)
esp_wins = np.sum(goals_sim_esp > goals_sim_arg)
draws = np.sum(goals_sim_arg == goals_sim_esp)

#Calculates win/draw percetange rate 
arg_win_rate = (arg_wins / sim_num) * 100
esp_win_rate = (esp_wins / sim_num) * 100
draw_rate = (draws / sim_num) * 100


In [86]:
#This block calculates possesion % predictions  
#Extracts % rates for each team 
esp_poss_rate = model_df.loc['España', 'Posesión del balón (%)']
arg_poss_rate = model_df.loc['Argentina', 'Posesión del balón (%)']
total_pos = esp_poss_rate + arg_poss_rate
prob_pos_esp = (esp_poss_rate / total_pos) * 100
prob_pos_arg = (arg_poss_rate / total_pos) * 100

In [88]:
print("="*40)
print("📋 FINAL PREDICTIONS")
print("="*40)
print(f"Match winner (Monte Carlo):")
print(f" 🇦🇷 Argentina: {arg_win_rate:.2f}% | 🤝 Draw: {draw_rate:.2f}% | 🇪🇸 España: {esp_win_rate:.2f}%")
print("-"*40)
print(f"Expected possesion %:")
print(f" 🇪🇸 España: {prob_pos_esp:.2f}% | 🇦🇷 Argentina: {prob_pos_arg:.2f}%")
print("="*40)

📋 FINAL PREDICTIONS
Match winner (Monte Carlo):
 🇦🇷 Argentina: 33.28% | 🤝 Draw: 20.99% | 🇪🇸 España: 45.73%
----------------------------------------
Expected possesion %:
 🇪🇸 España: 51.33% | 🇦🇷 Argentina: 48.67%


In [91]:
#This block prepares data to be exported to power bi in csv docs 
# 1. Goal Distribution Table (Poisson Curves from 0 to 5 goals)
goals_data = {
    "Goals": list(range(6)) * 2,
    "Team": ["Spain"] * 6 + ["Argentina"] * 6,
    "Probability": prob_goals_esp + prob_goals_arg
}
df_bi_goals = pd.DataFrame(goals_data)
df_bi_goals.to_csv("bi_goal_distribution.csv", index=False)

# 2. Corners Table (Casino Lines from 6 to 10)
lines = list(corner_probs.keys())
prob_under = [corner_probs[line]['Under'] for line in lines]
prob_over = [corner_probs[line]['Over'] for line in lines]

df_bi_corners = pd.DataFrame({
    "Casino_Line": lines,
    "Prob_Under": prob_under,
    "Prob_Over": prob_over
})
df_bi_corners.to_csv("bi_corner_lines.csv", index=False)
df_bi_corners.to_csv("bi_corner_lines.csv", index=False)

# 3. Monte Carlo Results Table (Global Rates & Balanced Possession)
mc_data = {
    "Metric": ["Win Argentina", "Win Spain", "Draw (90 Min)", "Possession Argentina", "Possession Spain"],
    "Percentage_Value": [arg_win_rate, esp_win_rate, draw_rate, prob_pos_esp, prob_pos_arg]
}
df_bi_montecarlo = pd.DataFrame(mc_data)
df_bi_montecarlo.to_csv("bi_montecarlo_insights.csv", index=False)